In [ ]:
# ===============================================================
# --- Checking Cuda Availability ---
# ===============================================================

import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
# ===============================================================
# --- Install Required Packages ---
# ===============================================================

!pip install mulaconf
!pip install xgboost

In [ ]:
# ===========================================================
# --- Example ----
# ===========================================================
import pandas as pd
import numpy as np
import torch
import os

from tqdm import tqdm
import requests

from sklearn.model_selection import train_test_split

# Dataset: "yeast" or "tmc2007_500"
dataset = "yeast"

# Download data from data folder on GitHub
data_dir = os.path.join(os.getcwd(), f"data/{dataset}")
os.makedirs(data_dir, exist_ok=True)

base_url = f"https://raw.githubusercontent.com/k-kostas/MuLaConf/main/examples/data/{dataset}"

x_output_path = os.path.join(data_dir, f"X_{dataset}.csv")
y_output_path = os.path.join(data_dir, f"y_{dataset}.csv")

# Download X data
print(f"Downloading X_{dataset}.csv...")
response_x = requests.get(f"{base_url}/X_{dataset}.csv", allow_redirects=True, timeout=60)
response_x.raise_for_status()
with open(x_output_path, "wb") as f:
    f.write(response_x.content)

# Download y data
print(f"Downloading y_{dataset}.csv...")
response_y = requests.get(f"{base_url}/y_{dataset}.csv", allow_redirects=True, timeout=60)
response_y.raise_for_status()
with open(y_output_path, "wb") as f:
    f.write(response_y.content)

out_dir = f"{dataset}_experiments"
os.makedirs(out_dir, exist_ok=True)

print(f"Loading {dataset} dataset...")
X = pd.read_csv(x_output_path).values
y = pd.read_csv(y_output_path).values
print(f"X shape: {X.shape}, y shape: {y.shape}")

# First, separate out the Test set (10%)
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.1, random_state=42)

# Then, split the remaining data into Proper Train and Calibration (30%)
X_train, X_calib, y_train, y_calib = train_test_split(X_temp, y_temp, test_size=0.3, random_state=42)


In [ ]:
import logging

logging.basicConfig(
    level=logging.DEBUG,
    format="%(asctime)s [%(levelname)s] %(message)s",
    force=True
)


from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

from mulaconf.icp_wrapper import ICPWrapper

base_model = MultiOutputClassifier(RandomForestClassifier(n_estimators=10))
wrapper = ICPWrapper(base_model,
                     measure='mahalanobis',
                     weight_hamming=2.0,
                     weight_cardinality=1.5,
                     device='cuda')

In [ ]:
wrapper.fit(X_train, y_train)

In [ ]:
wrapper.calibrate(X_calib, y_calib)

In [ ]:
# Update parameters directly after calibration
wrapper.weight_hamming = 3.0
wrapper.weight_cardinality = 2.5
wrapper.calibrate()

In [ ]:
prediction_obj = wrapper.predict(X_test)

In [ ]:
# Extract prediction sets for one or multiple significance levels.
prediction_regions = prediction_obj(significance_level=0.1)
print(prediction_regions[0])

# print(prediction_obj.p_values[0])

In [ ]:
#  Evaluation metrics
metrics = prediction_obj.evaluate(
    return_true_label_p_value = False,
    return_coverage=True,
    return_n_criterion=True,
    return_s_criterion=True,
    return_observed_fuzziness=True,
    return_observed_excess=True,
    true_labelsets=y_test,
    significance_level=0.1,
)

print(metrics)

In [ ]:
# Update parameters directly after prediction
wrapper.measure = 'norm'
wrapper.weight_hamming = 1.0
wrapper.weight_cardinality = 0.5

# Recalibration happens automatically on the fly, so you can call the predict() method immediately.
updated_prediction_obj = wrapper.predict(X_test)

In [ ]:
# Extract prediction sets for one or multiple significance levels.
updated_prediction_regions = updated_prediction_obj(significance_level=0.1)

print(updated_prediction_regions[0])

# print(updated_prediction_obj.p_values[0])

In [ ]:
#  Evaluation metrics for the updated predict
metrics = updated_prediction_obj.evaluate(
    return_true_label_p_value = False,
    return_coverage=True,
    return_n_criterion=True,
    return_s_criterion=True,
    return_observed_fuzziness=True,
    return_observed_excess=True,
    true_labelsets=y_test,
    significance_level=0.1,
)

print(metrics)